# 05 -- Training GPT-2: Next-Token Prediction

This notebook walks through how GPT-2 is trained from scratch using the **causal language modeling** objective. We will:

1. Re-implement a self-contained tiny GPT-2 model inline (no external imports)
2. Explain the **next-token prediction** objective and how inputs align with targets
3. Implement and compare **cross-entropy loss** -- the training signal for language models
4. Understand **teacher forcing** -- the training strategy that makes autoregressive training tractable
5. Build a toy byte-level dataset and a simple training loop
6. Visualize loss curves, gradient norms, and perplexity during training
7. Run ablations on learning rate and model size

Everything runs on CPU in seconds. The model is deliberately tiny:
- `vocab_size=256` (byte-level, one token per ASCII character)
- `embed_dim=64`, `n_heads=4`, `n_layers=2`, `max_seq_len=128`

In [ ]:
%matplotlib inline

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: cpu (tiny model -- no GPU needed)")

---
## 1. Self-Contained GPT-2 Model

We re-implement the full GPT-2 architecture inline so this notebook is completely self-contained.
The architecture follows the standard decoder-only transformer:

- Token + positional embeddings
- N transformer blocks, each with causal multi-head attention + FFN
- Pre-norm (LayerNorm before attention and FFN)
- Final LayerNorm + linear head projecting to vocabulary logits

In [ ]:
class MultiHeadAttention(nn.Module):
    """Causal multi-head self-attention."""

    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** 0.5

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention with causal mask
        scores = (q @ k.transpose(-2, -1)) / self.scale
        causal_mask = torch.tril(torch.ones(T, T, device=x.device)).unsqueeze(0).unsqueeze(0)
        scores = scores.masked_fill(causal_mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)

        out = (weights @ v).transpose(1, 2).reshape(B, T, C)
        return self.out_proj(out)


class TransformerBlock(nn.Module):
    """Pre-norm transformer block: LN -> Attention -> Residual -> LN -> FFN -> Residual."""

    def __init__(self, embed_dim, num_heads, ffn_dim):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, embed_dim),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class GPT2(nn.Module):
    """Tiny GPT-2 model for next-token prediction."""

    def __init__(self, vocab_size=256, embed_dim=64, num_heads=4,
                 num_layers=2, ffn_dim=256, max_seq_len=128):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(max_seq_len, embed_dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ffn_dim)
            for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.max_seq_len = max_seq_len

    def forward(self, idx):
        """idx: (batch, seq_len) -> logits: (batch, seq_len, vocab_size)"""
        B, T = idx.shape
        x = self.token_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters())


# Quick sanity check
model = GPT2()
dummy = torch.randint(0, 256, (2, 32))
logits = model(dummy)
print(f"Model parameters: {model.count_parameters():,}")
print(f"Input shape:  {dummy.shape}")
print(f"Output shape: {logits.shape}  (batch, seq_len, vocab_size)")

---
## 2. The Next-Token Prediction Objective

GPT-2 is trained with **causal language modeling**: given a sequence of tokens, the model predicts the next token at each position. The causal mask ensures position `i` can only attend to positions `0..i`, so each prediction is autoregressive.

Concretely, given tokens `[t0, t1, t2, t3, t4]`:

| Position | Input token | Model sees | Target |
|----------|------------|------------|--------|
| 0        | t0         | t0         | t1     |
| 1        | t1         | t0, t1     | t2     |
| 2        | t2         | t0, t1, t2 | t3     |
| 3        | t3         | t0..t3     | t4     |

The input is `tokens[:-1]` and the target is `tokens[1:]` -- shifted by one position.

In [ ]:
# Demonstrate input-target alignment
example_text = "hello world"
tokens = [ord(c) for c in example_text]

inputs  = tokens[:-1]  # Feed these to the model
targets = tokens[1:]   # Model should predict these

print("Input-Target Alignment for next-token prediction:")
print("=" * 55)
print(f"{'Position':<10} {'Input char':<12} {'Input ID':<10} {'Target char':<12} {'Target ID'}")
print("-" * 55)
for i, (inp, tgt) in enumerate(zip(inputs, targets)):
    inp_char = repr(chr(inp))
    tgt_char = repr(chr(tgt))
    print(f"{i:<10} {inp_char:<12} {inp:<10} {tgt_char:<12} {tgt}")

In [ ]:
# Visualize the input-target alignment
fig, ax = plt.subplots(figsize=(12, 3))

full_text = "To be or not to be"
chars = list(full_text)
n = len(chars)

# Draw input boxes (blue)
for i in range(n - 1):
    rect = plt.Rectangle((i, 1.5), 0.9, 0.8, facecolor='#4C72B0', edgecolor='black', alpha=0.8)
    ax.add_patch(rect)
    display_char = chars[i] if chars[i] != ' ' else '_'
    ax.text(i + 0.45, 1.9, display_char, ha='center', va='center', fontsize=11,
            color='white', fontweight='bold')

# Draw target boxes (red)
for i in range(n - 1):
    rect = plt.Rectangle((i, 0.2), 0.9, 0.8, facecolor='#C44E52', edgecolor='black', alpha=0.8)
    ax.add_patch(rect)
    display_char = chars[i + 1] if chars[i + 1] != ' ' else '_'
    ax.text(i + 0.45, 0.6, display_char, ha='center', va='center', fontsize=11,
            color='white', fontweight='bold')

    # Arrow from input to target
    ax.annotate('', xy=(i + 0.45, 1.05), xytext=(i + 0.45, 1.45),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

ax.text(-1.5, 1.9, 'Input:', ha='center', va='center', fontsize=12, fontweight='bold', color='#4C72B0')
ax.text(-1.5, 0.6, 'Target:', ha='center', va='center', fontsize=12, fontweight='bold', color='#C44E52')

ax.set_xlim(-2.5, n)
ax.set_ylim(-0.2, 2.8)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Next-Token Prediction: Input and Target Alignment', fontsize=14, pad=10)
plt.tight_layout()
plt.show()

The diagram above shows how, at each position, the model receives the current token as input and must predict the next token as the target. The entire sequence is processed in parallel during training (thanks to the causal mask), but each position only has access to past context.

---
## 3. Cross-Entropy Loss

The model outputs a vector of **logits** (one per vocabulary token) at each position. We convert these to probabilities with softmax, then measure how far the predicted distribution is from the true next token using **cross-entropy loss**:

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N} \log p(t_i | t_1, \ldots, t_{i-1})$$

Where $p(t_i | \ldots)$ is the model's predicted probability for the correct next token.

**Perplexity** is the exponentiated loss: $\text{PPL} = e^{\mathcal{L}}$. It can be interpreted as the effective number of equally likely tokens the model is choosing from. Lower is better -- a perplexity of 1 means the model is perfectly certain.

In [ ]:
# Manual cross-entropy implementation
def manual_cross_entropy(logits, targets):
    """
    logits: (batch*seq_len, vocab_size) -- raw model outputs
    targets: (batch*seq_len,) -- ground-truth token IDs
    """
    # Step 1: Convert logits to log-probabilities (numerically stable)
    log_probs = logits - torch.logsumexp(logits, dim=-1, keepdim=True)

    # Step 2: Gather the log-probability assigned to the correct class
    target_log_probs = log_probs[torch.arange(len(targets)), targets]

    # Step 3: Negate and average
    loss = -target_log_probs.mean()
    return loss


# Compare manual vs PyTorch implementation
torch.manual_seed(0)
test_logits = torch.randn(8, 256)   # 8 predictions over 256 vocab
test_targets = torch.randint(0, 256, (8,))

manual_loss = manual_cross_entropy(test_logits, test_targets)
pytorch_loss = F.cross_entropy(test_logits, test_targets)

print(f"Manual cross-entropy:  {manual_loss.item():.6f}")
print(f"PyTorch cross-entropy: {pytorch_loss.item():.6f}")
print(f"Difference:            {abs(manual_loss.item() - pytorch_loss.item()):.2e}")
print(f"\nPerplexity: exp({pytorch_loss.item():.4f}) = {torch.exp(pytorch_loss).item():.2f}")
print(f"Random baseline perplexity (vocab=256): {256.0:.2f}")

In [ ]:
# Visualize how loss relates to predicted probability
probs = np.linspace(0.001, 1.0, 500)
losses = -np.log(probs)
perplexities = np.exp(losses)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(probs, losses, 'b-', linewidth=2)
ax1.axhline(y=-np.log(1/256), color='r', linestyle='--', alpha=0.7, label=f'Random guess (1/256)')
ax1.set_xlabel('Predicted probability of correct token', fontsize=11)
ax1.set_ylabel('Cross-entropy loss', fontsize=11)
ax1.set_title('Cross-Entropy Loss vs. Predicted Probability', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 7)

# Perplexity vs loss
loss_range = np.linspace(0.01, 6.0, 500)
ppl_range = np.exp(loss_range)
ax2.plot(loss_range, ppl_range, 'b-', linewidth=2)
ax2.axvline(x=np.log(256), color='r', linestyle='--', alpha=0.7, label='Random (loss=5.55)')
ax2.axhline(y=256, color='r', linestyle=':', alpha=0.5)
ax2.set_xlabel('Cross-entropy loss', fontsize=11)
ax2.set_ylabel('Perplexity', fontsize=11)
ax2.set_title('Perplexity = exp(Loss)', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 400)

plt.tight_layout()
plt.show()

**Observations:**
- Cross-entropy loss is the negative log of the predicted probability for the correct token. When the model assigns probability 1.0 to the correct token, the loss is 0.
- For a random model over 256 tokens, the expected loss is $\log(256) \approx 5.55$ and perplexity is 256.
- Perplexity grows exponentially with loss -- even small loss reductions mean large perplexity improvements.

---
## 4. Teacher Forcing

During training, GPT-2 uses **teacher forcing**: the model always receives the ground-truth tokens as input, regardless of what it would have predicted.

**Why teacher forcing?**
- Without it ("free-running"), early errors compound: a wrong prediction at position 3 feeds garbage into position 4, which makes position 5 even worse.
- Teacher forcing provides a stable training signal at every position in parallel.
- The causal mask already prevents the model from "cheating" -- position `i` cannot see position `i+1`.

**The tradeoff:** At inference time, the model must use its own predictions as input (no teacher), creating an **exposure bias** -- the model never saw its own mistakes during training. In practice, this works well enough for large models.

In [ ]:
# Demonstrate teacher forcing vs free-running generation

model_demo = GPT2()
model_demo.eval()

prompt = "to be or"
prompt_ids = torch.tensor([[ord(c) for c in prompt]])

# Teacher forcing: feed all ground-truth tokens at once, get all predictions in parallel
with torch.no_grad():
    logits_tf = model_demo(prompt_ids)  # (1, seq_len, vocab_size)
    preds_tf = logits_tf.argmax(dim=-1)[0]  # Greedy prediction at each position

print("Teacher Forcing (parallel, ground-truth input):")
print(f"  Input:       {[chr(t) for t in prompt_ids[0].tolist()]}")
print(f"  Predictions: {[chr(t) if 32 <= t < 127 else '?' for t in preds_tf.tolist()]}")
print()

# Free-running: feed own predictions autoregressively
generated = list(prompt_ids[0].tolist())
with torch.no_grad():
    for _ in range(10):  # Generate 10 more tokens
        inp = torch.tensor([generated[-model_demo.max_seq_len:]])
        logits_fr = model_demo(inp)
        next_token = logits_fr[0, -1].argmax().item()
        generated.append(next_token)

free_run_text = ''.join(chr(t) if 32 <= t < 127 else '?' for t in generated)
print("Free-Running (autoregressive, own predictions as input):")
print(f"  Output: {repr(free_run_text)}")
print()
print("Note: The untrained model produces garbage. After training, free-running")
print("generation will produce coherent text (at least for our toy dataset).")

---
## 5. Toy Dataset

We create a minimal byte-level dataset from a short repeated text. Each character is encoded as its ASCII byte value (0-255), so `vocab_size=256` covers everything we need.

The dataset is intentionally tiny (a few hundred tokens) so training completes in seconds on CPU.

In [ ]:
# Build a tiny byte-level dataset
text = "To be or not to be that is the question " * 12
data = torch.tensor([ord(c) for c in text], dtype=torch.long)

print(f"Text length: {len(text)} characters")
print(f"Tensor shape: {data.shape}")
print(f"Unique bytes: {len(torch.unique(data))}")
print(f"First 60 chars: {repr(text[:60])}")
print(f"First 20 byte values: {data[:20].tolist()}")

In [ ]:
# Dataset: create (input, target) pairs by slicing windows from the data

SEQ_LEN = 32
BATCH_SIZE = 8


def get_batch(data, seq_len=SEQ_LEN, batch_size=BATCH_SIZE):
    """Sample a random batch of (input, target) pairs."""
    max_start = len(data) - seq_len - 1
    starts = torch.randint(0, max_start, (batch_size,))
    inputs = torch.stack([data[s : s + seq_len] for s in starts])
    targets = torch.stack([data[s + 1 : s + seq_len + 1] for s in starts])
    return inputs, targets


# Show one batch
inp, tgt = get_batch(data)
print(f"Input batch shape:  {inp.shape}  (batch_size, seq_len)")
print(f"Target batch shape: {tgt.shape}  (batch_size, seq_len)")
print()
print("Example pair (first sample):")
inp_text = ''.join(chr(c) for c in inp[0].tolist())
tgt_text = ''.join(chr(c) for c in tgt[0].tolist())
print(f"  Input:  {repr(inp_text)}")
print(f"  Target: {repr(tgt_text)}")
print(f"  (Target is Input shifted right by 1 position)")

---
## 6. Training Loop

Now we put it all together. The training loop:

1. Sample a batch of `(input, target)` pairs
2. Forward pass: compute logits
3. Compute cross-entropy loss between predicted logits and target tokens
4. Backward pass: compute gradients
5. Optimizer step: update parameters

We track loss, perplexity, and gradient norms throughout training.

In [ ]:
def compute_grad_norms(model):
    """Compute gradient norm for each named parameter group."""
    norms = {}
    for name, param in model.named_parameters():
        if param.grad is not None:
            norms[name] = param.grad.norm().item()
    return norms


def compute_layer_grad_norms(model):
    """Compute average gradient norm per transformer block."""
    layer_norms = {}
    # Embedding layers
    emb_grads = []
    for name, param in model.named_parameters():
        if param.grad is None:
            continue
        if 'emb' in name:
            emb_grads.append(param.grad.norm().item())
        for i in range(len(model.blocks)):
            if f'blocks.{i}.' in name:
                key = f'block_{i}'
                if key not in layer_norms:
                    layer_norms[key] = []
                layer_norms[key].append(param.grad.norm().item())
        if 'head' in name or 'ln_f' in name:
            if 'output' not in layer_norms:
                layer_norms['output'] = []
            layer_norms['output'].append(param.grad.norm().item())

    if emb_grads:
        layer_norms['embeddings'] = emb_grads

    return {k: np.mean(v) for k, v in layer_norms.items()}


print("Helper functions defined.")

In [ ]:
# Training loop
torch.manual_seed(42)

model = GPT2(vocab_size=256, embed_dim=64, num_heads=4, num_layers=2, ffn_dim=256, max_seq_len=128)
optimizer = optim.Adam(model.parameters(), lr=3e-4)

NUM_STEPS = 100

# Tracking
loss_history = []
ppl_history = []
grad_norm_history = []  # Total grad norm
layer_grad_history = []  # Per-layer grad norms

model.train()
for step in range(NUM_STEPS):
    inp, tgt = get_batch(data)

    logits = model(inp)  # (batch, seq_len, vocab_size)

    # Reshape for cross-entropy: (batch*seq_len, vocab_size) vs (batch*seq_len,)
    loss = F.cross_entropy(logits.view(-1, 256), tgt.view(-1))

    optimizer.zero_grad()
    loss.backward()

    # Track gradient norms before optimizer step
    total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    grad_norm_history.append(total_norm.item())
    layer_grad_history.append(compute_layer_grad_norms(model))

    optimizer.step()

    loss_val = loss.item()
    ppl_val = np.exp(loss_val)
    loss_history.append(loss_val)
    ppl_history.append(ppl_val)

    if step % 10 == 0 or step == NUM_STEPS - 1:
        print(f"Step {step:3d} | Loss: {loss_val:.4f} | PPL: {ppl_val:.1f} | Grad norm: {total_norm:.4f}")

print(f"\nTraining complete. Final loss: {loss_history[-1]:.4f}, Final PPL: {ppl_history[-1]:.1f}")

---
## 7. Training Visualizations

Let us visualize the training dynamics: loss curve, perplexity, and gradient norms.

In [ ]:
# Loss curve
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss
axes[0].plot(loss_history, 'b-', linewidth=1.5, alpha=0.8)
axes[0].axhline(y=np.log(256), color='r', linestyle='--', alpha=0.5, label='Random baseline')
axes[0].set_xlabel('Training step')
axes[0].set_ylabel('Cross-entropy loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Perplexity
axes[1].plot(ppl_history, 'g-', linewidth=1.5, alpha=0.8)
axes[1].axhline(y=256, color='r', linestyle='--', alpha=0.5, label='Random baseline (256)')
axes[1].set_xlabel('Training step')
axes[1].set_ylabel('Perplexity')
axes[1].set_title('Perplexity (lower = better)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Gradient norms
axes[2].plot(grad_norm_history, 'm-', linewidth=1.5, alpha=0.8)
axes[2].set_xlabel('Training step')
axes[2].set_ylabel('Total gradient norm')
axes[2].set_title('Gradient Norm (after clipping)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Per-layer gradient norms over training
layer_names = sorted(layer_grad_history[0].keys())

fig, ax = plt.subplots(figsize=(12, 5))

for layer_name in layer_names:
    values = [h.get(layer_name, 0) for h in layer_grad_history]
    ax.plot(values, label=layer_name, linewidth=1.5, alpha=0.8)

ax.set_xlabel('Training step', fontsize=11)
ax.set_ylabel('Average gradient norm', fontsize=11)
ax.set_title('Gradient Norms by Layer', fontsize=13)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Observations:**
- The loss drops rapidly from the random baseline (~5.5) as the model learns basic character frequencies.
- Perplexity mirrors the loss curve but on an exponential scale, making early improvements look dramatic.
- Gradient norms are highest in the first few steps (the model is far from the optimum) and stabilize as training progresses.
- Different layers may have different gradient magnitudes -- the embedding and output layers often have larger gradients.

---
## 8. Verifying What the Model Learned

Let us see if the trained model can generate text that resembles our training data.

In [ ]:
# Generate text from the trained model
def generate(model, prompt_text, max_new_tokens=60, temperature=0.8):
    """Autoregressive generation with temperature sampling."""
    model.eval()
    tokens = [ord(c) for c in prompt_text]
    input_ids = torch.tensor([tokens], dtype=torch.long)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Truncate to max_seq_len
            inp = input_ids[:, -model.max_seq_len:]
            logits = model(inp)
            next_logits = logits[0, -1] / temperature
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)

    output = ''.join(chr(t) if 32 <= t < 127 else '?' for t in input_ids[0].tolist())
    return output


print("Generated text (prompt = 'To be'):")
print("-" * 60)
torch.manual_seed(42)
print(generate(model, "To be"))
print()
print("Generated text (prompt = 'not'):")
print("-" * 60)
torch.manual_seed(123)
print(generate(model, "not"))

Even with only 100 training steps on a tiny model and tiny dataset, the model picks up the repeated structure and vocabulary of the training text. It may not be perfect, but it clearly learns the dominant patterns.

---
## 9. Ablation: Learning Rate Impact

Learning rate is arguably the most important hyperparameter. Let us compare three values:
- **Too high** (`1e-2`): may diverge or oscillate
- **Just right** (`3e-4`): standard Adam default for transformers
- **Too low** (`1e-5`): converges slowly, may not learn much in 100 steps

In [ ]:
# Learning rate ablation
learning_rates = {'lr=1e-2': 1e-2, 'lr=3e-4': 3e-4, 'lr=1e-5': 1e-5}
lr_results = {}

for name, lr in learning_rates.items():
    torch.manual_seed(42)
    m = GPT2(vocab_size=256, embed_dim=64, num_heads=4, num_layers=2, ffn_dim=256, max_seq_len=128)
    opt = optim.Adam(m.parameters(), lr=lr)
    losses = []

    m.train()
    for step in range(100):
        inp, tgt = get_batch(data)
        logits = m(inp)
        loss = F.cross_entropy(logits.view(-1, 256), tgt.view(-1))
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=1.0)
        opt.step()
        losses.append(loss.item())

    lr_results[name] = losses
    print(f"{name}: final loss = {losses[-1]:.4f}, final PPL = {np.exp(losses[-1]):.1f}")

print("\nAll learning rate runs complete.")

In [ ]:
# Plot learning rate comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

colors = {'lr=1e-2': '#C44E52', 'lr=3e-4': '#4C72B0', 'lr=1e-5': '#55A868'}

for name, losses in lr_results.items():
    ax1.plot(losses, label=name, color=colors[name], linewidth=2, alpha=0.85)

ax1.axhline(y=np.log(256), color='gray', linestyle='--', alpha=0.4, label='Random baseline')
ax1.set_xlabel('Training step', fontsize=11)
ax1.set_ylabel('Cross-entropy loss', fontsize=11)
ax1.set_title('Loss Curves: Learning Rate Comparison', fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Perplexity comparison
for name, losses in lr_results.items():
    ppls = [np.exp(l) for l in losses]
    ax2.plot(ppls, label=name, color=colors[name], linewidth=2, alpha=0.85)

ax2.axhline(y=256, color='gray', linestyle='--', alpha=0.4, label='Random baseline')
ax2.set_xlabel('Training step', fontsize=11)
ax2.set_ylabel('Perplexity', fontsize=11)
ax2.set_title('Perplexity Curves: Learning Rate Comparison', fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

**Learning rate findings:**
- `lr=1e-2` (too high): The loss may drop fast initially but can become unstable, oscillate, or even diverge.
- `lr=3e-4` (standard): Steady, reliable convergence. This is close to the default used in practice for Adam with transformers.
- `lr=1e-5` (too low): The model barely learns anything in 100 steps. Given enough time it would converge, but it wastes compute.

In practice, real GPT-2 training uses learning rate warmup + cosine decay schedules to get the best of both worlds.

---
## 10. Ablation: Model Size

Does a bigger model converge faster on our toy dataset? We compare:
- **Tiny**: `embed_dim=32, num_layers=1, num_heads=2, ffn_dim=128`
- **Small**: `embed_dim=64, num_layers=2, num_heads=4, ffn_dim=256`

In [ ]:
# Model size ablation
model_configs = {
    'tiny (d=32, L=1)': dict(vocab_size=256, embed_dim=32, num_heads=2, num_layers=1, ffn_dim=128, max_seq_len=128),
    'small (d=64, L=2)': dict(vocab_size=256, embed_dim=64, num_heads=4, num_layers=2, ffn_dim=256, max_seq_len=128),
}

size_results = {}

for name, config in model_configs.items():
    torch.manual_seed(42)
    m = GPT2(**config)
    n_params = m.count_parameters()
    opt = optim.Adam(m.parameters(), lr=3e-4)
    losses = []

    m.train()
    for step in range(100):
        inp, tgt = get_batch(data)
        logits = m(inp)
        loss = F.cross_entropy(logits.view(-1, 256), tgt.view(-1))
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=1.0)
        opt.step()
        losses.append(loss.item())

    size_results[name] = losses
    print(f"{name} ({n_params:,} params): final loss = {losses[-1]:.4f}, final PPL = {np.exp(losses[-1]):.1f}")

print("\nAll model size runs complete.")

In [ ]:
# Plot model size comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

colors_size = {'tiny (d=32, L=1)': '#DD8452', 'small (d=64, L=2)': '#4C72B0'}

for name, losses in size_results.items():
    ax1.plot(losses, label=name, color=colors_size[name], linewidth=2, alpha=0.85)

ax1.axhline(y=np.log(256), color='gray', linestyle='--', alpha=0.4, label='Random baseline')
ax1.set_xlabel('Training step', fontsize=11)
ax1.set_ylabel('Cross-entropy loss', fontsize=11)
ax1.set_title('Loss: Tiny vs Small Model', fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Perplexity
for name, losses in size_results.items():
    ppls = [np.exp(l) for l in losses]
    ax2.plot(ppls, label=name, color=colors_size[name], linewidth=2, alpha=0.85)

ax2.axhline(y=256, color='gray', linestyle='--', alpha=0.4, label='Random baseline')
ax2.set_xlabel('Training step', fontsize=11)
ax2.set_ylabel('Perplexity', fontsize=11)
ax2.set_title('Perplexity: Tiny vs Small Model', fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

**Model size findings:**
- The larger model (64d, 2 layers) converges faster and typically reaches a lower final loss.
- The tiny model (32d, 1 layer) has fewer parameters and thus less capacity -- it may plateau at a higher loss.
- For our very small dataset, the difference is moderate since even the tiny model has enough capacity for simple patterns.
- In real-world settings, the gap grows dramatically: larger models learn richer representations from bigger datasets.

---
## 11. Zooming In: Loss Landscape Around a Training Point

To build intuition about optimization, let us visualize how the loss changes as we perturb model parameters in a random direction from the current optimum.

In [ ]:
# 1D loss landscape slice along a random direction
model.eval()

# Get a fixed batch
torch.manual_seed(0)
fixed_inp, fixed_tgt = get_batch(data)

# Save original parameters
orig_params = {name: p.data.clone() for name, p in model.named_parameters()}

# Random direction (normalized)
direction = {}
for name, p in model.named_parameters():
    d = torch.randn_like(p)
    d = d / d.norm() * p.data.norm()  # Scale to match parameter magnitude
    direction[name] = d

# Sweep along direction
alphas = np.linspace(-0.5, 0.5, 50)
landscape_losses = []

with torch.no_grad():
    for alpha in alphas:
        # Perturb parameters
        for name, p in model.named_parameters():
            p.data = orig_params[name] + alpha * direction[name]

        logits = model(fixed_inp)
        loss = F.cross_entropy(logits.view(-1, 256), fixed_tgt.view(-1))
        landscape_losses.append(loss.item())

    # Restore original parameters
    for name, p in model.named_parameters():
        p.data = orig_params[name]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(alphas, landscape_losses, 'b-', linewidth=2)
ax.axvline(x=0, color='r', linestyle='--', alpha=0.7, label='Current parameters')
ax.set_xlabel('Perturbation magnitude (alpha)', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('1D Slice of Loss Landscape', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The trained model sits near a local minimum of the loss landscape. Moving in any random direction increases the loss, confirming that the optimizer has found a good region of parameter space.

---
## 12. Token Probability Distribution After Training

Let us look at what probability distribution the model has learned for next-token prediction.

In [ ]:
# Examine predicted distributions for specific contexts
model.eval()

contexts = [
    "To be",
    "not to",
    "the que",
    "that is",
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for idx, ctx in enumerate(contexts):
    tokens = torch.tensor([[ord(c) for c in ctx]], dtype=torch.long)
    with torch.no_grad():
        logits = model(tokens)
        probs = F.softmax(logits[0, -1], dim=-1).numpy()

    # Show top 10 predictions
    top_k = 10
    top_indices = np.argsort(probs)[-top_k:][::-1]
    top_probs = probs[top_indices]
    top_chars = [chr(i) if 32 <= i < 127 else f'[{i}]' for i in top_indices]

    ax = axes[idx]
    bars = ax.barh(range(top_k), top_probs[::-1], color='#4C72B0', alpha=0.8)
    ax.set_yticks(range(top_k))
    ax.set_yticklabels([repr(c) for c in top_chars[::-1]], fontsize=9)
    ax.set_xlabel('Probability', fontsize=10)
    ax.set_title(f'After "{ctx}" -> predict next', fontsize=11)
    ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('Learned Next-Token Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

The model has learned the character patterns in the training text. After "To be" it should predict a space; after "the que" it should predict 's' (from "question"). The distributions reflect what the model actually saw during training.

---
## 13. Summary: The Full Training Picture

Let us consolidate all metrics from our main training run into a single dashboard.

In [ ]:
# Summary dashboard
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Loss
axes[0, 0].plot(loss_history, 'b-', linewidth=1.5)
axes[0, 0].axhline(y=np.log(256), color='r', linestyle='--', alpha=0.5, label='Random baseline')
axes[0, 0].fill_between(range(len(loss_history)), loss_history, np.log(256),
                         where=[l < np.log(256) for l in loss_history], alpha=0.15, color='blue')
axes[0, 0].set_xlabel('Step')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Perplexity (log scale)
axes[0, 1].semilogy(ppl_history, 'g-', linewidth=1.5)
axes[0, 1].axhline(y=256, color='r', linestyle='--', alpha=0.5, label='Random (256)')
axes[0, 1].axhline(y=1, color='gray', linestyle=':', alpha=0.5, label='Perfect (1)')
axes[0, 1].set_xlabel('Step')
axes[0, 1].set_ylabel('Perplexity (log scale)')
axes[0, 1].set_title('Perplexity')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Gradient norms
axes[1, 0].plot(grad_norm_history, 'm-', linewidth=1.5, alpha=0.8)
axes[1, 0].set_xlabel('Step')
axes[1, 0].set_ylabel('Grad norm')
axes[1, 0].set_title('Total Gradient Norm')
axes[1, 0].grid(True, alpha=0.3)

# LR comparison summary (final loss bar chart)
lr_names = list(lr_results.keys())
lr_final_losses = [lr_results[n][-1] for n in lr_names]
bar_colors = [colors[n] for n in lr_names]
axes[1, 1].bar(lr_names, lr_final_losses, color=bar_colors, alpha=0.85)
axes[1, 1].axhline(y=np.log(256), color='gray', linestyle='--', alpha=0.4, label='Random baseline')
axes[1, 1].set_ylabel('Final loss')
axes[1, 1].set_title('Final Loss by Learning Rate')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.suptitle('GPT-2 Training Dashboard', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---
## Key Insights

1. **Next-token prediction is the core objective.** GPT-2 learns to predict `tokens[1:]` given `tokens[:-1]`. This single objective -- when applied to massive text corpora -- is sufficient to learn grammar, facts, reasoning, and more.

2. **Cross-entropy loss measures prediction quality.** It is the negative log-probability assigned to the correct next token. A random model over 256 byte tokens has loss ~5.55 (perplexity 256). Training drives this down.

3. **Teacher forcing makes training efficient.** By always feeding ground-truth tokens as input, the model receives a stable learning signal at every position in parallel. This is far more sample-efficient than free-running generation during training.

4. **Learning rate is critical.** Too high and training becomes unstable; too low and the model barely learns within a compute budget. The standard choice for Adam + transformers is around 1e-4 to 3e-4, often with warmup.

5. **Bigger models converge faster per step** (but cost more per step). This is the fundamental scaling law observation: given a fixed compute budget, there is an optimal model size.

6. **Gradient norms reveal training health.** Exploding gradients signal instability (often from a learning rate that is too high); vanishing gradients signal the model is stuck. Gradient clipping acts as a safety valve.

7. **Perplexity is the interpretable metric.** A perplexity of 10 means the model is as uncertain as choosing uniformly among 10 options at each step. Real GPT-2 (1.5B parameters) achieved perplexity ~35 on WebText, meaning it narrowed 50,000+ token options down to about 35 equally likely candidates at each position.

8. **Everything here scales.** Replace the 480-character toy dataset with 40 GB of internet text, replace the 100K-parameter model with 1.5 billion parameters, and train for 300,000 steps instead of 100 -- the same code structure produces GPT-2.